In [1]:
PIPELINE_RUN_ID = ''

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 3, Finished, Available, Finished, False)

In [2]:
%run ./NB_Utils_Common

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 9, Finished, Available, Finished, True)

NB_Utils_Common: settings ready
NB_Utils_Common: locations ready
NB_Utils_Common: start_run() ready
NB_Utils_Common: OPS logging ready
NB_Utils_Common: dq_record(), dq_flush(), check_volume_change() ready
NB_Utils_Common loaded: start_run(), tpath(), record_table(), dq_record(), end_of_run_report(), end_session()


In [3]:
%run ./NB_Config_PROC

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 16, Finished, Available, Finished, True)

Config loaded: ['EKKO', 'EKPO', 'EKET', 'EKBE']
EA audit report tables added: EKES, EKKN, EKPA, EKAB
New tables added:EKPV, EBKN, EBAN


In [4]:
%run ./NB_Utils_Gold

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 17, Finished, Available, Finished, True)

NB_Utils_Gold loaded: silver(), gold(), drop_redundant(), sk_lookup(), write_gold(), stable_sk(), promote_to_gold(), profile_strings()


In [5]:
# ── Setup ──────────────────────────────────────────────────────────────────────
start_run('PROC_Order_Status_Check', 'CHECK')
fact = gold('Fact_PO_LineItem_Schedule')
ev   = gold('Fact_PO_Order_Status_Evidence')   # ▶ 15 Nov 2026: the rule flags and the flags behind them (one row per fact row)
print(f"Fact_PO_LineItem_Schedule: {fact.count():,} schedule lines; Fact_PO_Order_Status_Evidence: {ev.count():,} segments "
      f"(▶ 17 Nov 2026: more segments than lines; {ev.filter(F.col('Schedule_Line').isNull()).count():,} over-confirmed pieces have no line)")

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 18, Finished, Available, Finished, False)

Pipeline : PROC_Order_Status_Check
Run ID   : edf70372-e7c0-43c7-a6fd-e89a04dbaf87
Started  : 2026-11-19 21:28:32 UTC
Locations:
  bronze     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  silver     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  gold       dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  md_silver  dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  md_gold    dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  ops        dbo              abfss://548c95f1-2c7c-44e4-a26f-e6971f80339f@onelake.dfs.fabric.micros…
  sd_silver  dbo              abfss://303715d4-7108-4456-a1f2-bbd04c28cc0c@onelake.dfs.fabric.micros…
  xrate      Exchange_Rates   abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
Fact_PO_LineItem_Schedule: 6,914,717 schedule lines; Fa

In [6]:
# ── 1 · Distribution of the status ──────────────────────────────────────────────────────────────────────
total = fact.count()
print(f'Rows by status and rule ({total:,} rows):')
(fact.groupBy('Order_Status_Rule', 'Order_Status_Purchase').count().orderBy('Order_Status_Rule').show(60, truncate=False))   # ▶ 17 Nov 2026: one row per line
print('Segments behind the lines (evidence table), by rule and source:')
(ev.groupBy('Order_Status_Rule', 'Order_Status_Purchase', 'Source_Type').count().orderBy('Order_Status_Rule', 'Source_Type').show(60, truncate=False))
ordered = fact.filter(F.col('Order_Status_Rule') == 18).count()
print(f"Fall-through 'Ordered': {ordered:,} rows ({100.0 * ordered / total:.1f}%). A high share points at evidence that is not reaching the segments.")
# ▶ 17 Nov 2026: the fact is one row per schedule line; the pieces are in the evidence table
print(f"Confirmed pieces (evidence): {ev.filter(F.col('Source_Type') == 'EKES').count():,}; lines with more than one piece: "
      f"{fact.filter(F.col('Piece_Count') > 1).count():,} (their status is the lowest rule of their segments)")
print(f"Rows without a status: {fact.filter(F.col('Order_Status_Purchase').isNull()).count():,} (expected 0)")

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 19, Finished, Available, Finished, False)

Rows by status and rule (6,914,717 rows):
+-----------------+------------------------------+-------+
|Order_Status_Rule|Order_Status_Purchase         |count  |
+-----------------+------------------------------+-------+
|0                |Return item                   |24851  |
|1                |Cancelled partially           |197964 |
|1                |Cancelled completely          |247207 |
|2                |Closed by GR                  |6188490|
|3                |Closed by order creation or IR|103610 |
|4                |Blocked                       |1437   |
|5                |GR blocked stock (103)        |141    |
|8                |Service entry sheet created   |917    |
|9                |Goods issued                  |3402   |
|10               |Outbound delivery created     |1121   |
|11               |Shipping notification         |7709   |
|14               |Stock committed               |1829   |
|15               |Confirmed                     |24873  |
|16           

In [7]:
# ── 2 · The status equals the first true rule flag (re-derived here from the stored flags) ──────────────────
# ▶ 6 Nov 2026: flag positions map to rule numbers through OS_RULES (rule 0 added, rule 6 retired)
_nos = F.array(*[F.lit(n) for n, _ in OS_RULES if n != 18])
first_true = F.array_position(F.array(*OS_RULE_FLAGS), True).cast('int')               # 0 when no flag is true
check = (ev   # ▶ 15 Nov 2026: the flags are in the evidence table
             .withColumn('_expected', F.when(first_true == 0, 18).otherwise(F.element_at(_nos, first_true))))
wrong = check.filter(F.col('_expected') != F.col('Order_Status_Rule'))
print(f"Rows whose status is not the first true flag: {wrong.count():,} (expected 0)")
wrong.select('Purchasing_Document', 'Item', 'Source_Type', 'Source_Sequence', 'Order_Status_Rule', '_expected').show(10, truncate=False)

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 20, Finished, Available, Finished, False)

Rows whose status is not the first true flag: 0 (expected 0)
+-------------------+----+-----------+---------------+-----------------+---------+
|Purchasing_Document|Item|Source_Type|Source_Sequence|Order_Status_Rule|_expected|
+-------------------+----+-----------+---------------+-----------------+---------+
+-------------------+----+-----------+---------------+-----------------+---------+



In [8]:
# ── 3 · Rows per rule: flag true, status given, and hidden behind a higher rule ─────────────────────────
flag_true = (ev.select(F.posexplode(F.array(*OS_RULE_FLAGS)).alias('pos', 'hit')).filter('hit')
             .groupBy('pos').count().withColumn('Order_Status_Rule', F.element_at(F.array(*[F.lit(n) for n, _ in OS_RULES if n != 18]), F.col('pos') + 1)).select('Order_Status_Rule', F.col('count').alias('Flag_True')))
got_status = (fact.groupBy('Order_Status_Rule').agg(F.first('Order_Status_Purchase').alias('Status'), F.count('*').alias('Status_Given')))
(got_status.join(flag_true, 'Order_Status_Rule', 'left')
     .withColumn('Hidden_By_Higher_Rule', F.col('Flag_True') - F.col('Status_Given'))
     .orderBy('Order_Status_Rule').show(20, truncate=False))
print('Flag_True is empty for rule 18 (Ordered has no flag: it is what is left). A rule with Flag_True = 0 never fires in this data.')

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 21, Finished, Available, Finished, False)

+-----------------+------------------------------+------------+---------+---------------------+
|Order_Status_Rule|Status                        |Status_Given|Flag_True|Hidden_By_Higher_Rule|
+-----------------+------------------------------+------------+---------+---------------------+
|0                |Return item                   |24851       |24894    |43                   |
|1                |Cancelled completely          |445171      |447048   |1877                 |
|2                |Closed by GR                  |6188490     |6383318  |194828               |
|3                |Closed by order creation or IR|103610      |6550466  |6446856              |
|4                |Blocked                       |1437        |27845    |26408                |
|5                |GR blocked stock (103)        |141         |19190    |19049                |
|8                |Service entry sheet created   |917         |95563    |94646                |
|9                |Goods issued         

In [9]:
# ── 4 · EKES categories (EBTYP) that no status uses ──────────────────────────────────────────────────────
(silver('EKES')
    .withColumn('Status_Group',
        F.when(F.col('Confirmation_Category').isin(OS_CAT_RECEIPT),   'Receipt from supplier (rule 7)')
         .when(F.col('Confirmation_Category').isin(OS_CAT_SHIPPING),  'Shipping notification (rule 11)')
         .when(F.col('Confirmation_Category').isin(OS_CAT_CALL),      'Call from supplier (rule 12)')
         .when(F.col('Confirmation_Category').isin(OS_CAT_AVAILABLE), 'Goods available at supplier (rule 13)')
         .when(F.col('Confirmation_Category').isin(OS_CAT_CONFIRMED), 'Confirmed (rule 15)')
         .otherwise('-- not used by any status (question C1)'))
    .groupBy('Confirmation_Category', 'Status_Group').count().orderBy(F.desc('count')).show(50, truncate=False))

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 22, Finished, Available, Finished, False)

+---------------------+---------------------------------------+-------+
|Confirmation_Category|Status_Group                           |count  |
+---------------------+---------------------------------------+-------+
|ZS                   |Shipping notification (rule 11)        |1597973|
|AB                   |Confirmed (rule 15)                    |275580 |
|ZB                   |-- not used by any status (question C1)|141205 |
|GW                   |Shipping notification (rule 11)        |15165  |
|ZN                   |-- not used by any status (question C1)|4593   |
|                     |-- not used by any status (question C1)|732    |
|ZO                   |-- not used by any status (question C1)|147    |
|LA                   |-- not used by any status (question C1)|16     |
|PI                   |-- not used by any status (question C1)|1      |
|ZD                   |-- not used by any status (question C1)|1      |
+---------------------+---------------------------------------+-

In [10]:
# ── 5 · The existing grain and sums are unchanged ──────────────────────────────────────────────────────
s_eket = silver('EKET')
f_eket = fact.filter(F.col('Scheduled_Quantity').isNotNull())   # ▶ 17 Nov 2026: every fact row is a schedule line and carries its measures
n_s, n_f = s_eket.count(), f_eket.count()
once = f_eket.groupBy('PO_Schedule_SK').count().filter('count > 1').count()
print(f"EKET lines: Silver {n_s:,} | fact {n_f:,} | {'OK' if n_s == n_f else 'DIFFERENT'}; PO_Schedule_SK repeated: {once:,} (expected 0)")
a = s_eket.agg(F.sum('Scheduled_Quantity')).collect()[0][0]
b = f_eket.agg(F.sum('Scheduled_Quantity')).collect()[0][0]
print(f"Scheduled_Quantity: Silver {a} | fact {b} | {'OK' if a == b else 'DIFFERENT'}")
_dup = fact.groupBy('Purchasing_Document', 'Item', 'Schedule_Line').count().filter('count > 1').count()
print(f"Keys with more than one row: {_dup:,} (expected 0, one row per schedule line since 17 Nov 2026)")

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 23, Finished, Available, Finished, False)

EKET lines: Silver 6,914,717 | fact 6,914,717 | OK; PO_Schedule_SK repeated: 0 (expected 0)
Scheduled_Quantity: Silver 29910255807.388 | fact 29910255807.388 | OK
Keys with more than one row: 0 (expected 0, one row per schedule line since 17 Nov 2026)


In [11]:
# ── 6 · Conservation per item: open segment quantity vs order quantity minus net physical GR ──────────
_gr = (F.col('Transevent_Type') == '1') | (F.col('PO_History_Category') == 'E')
net_gr = (silver('EKBE').filter(_gr).groupBy('Purchasing_Document', 'Item')
          .agg(F.sum('Signed_Quantity').alias('_net_gr')))
seg_open = (fact.filter(F.col('Split_Number').isNotNull()).groupBy('Purchasing_Document', 'Item')
            .agg(F.sum('Segment_Quantity').cast('double').alias('_seg_open')))
cons = (silver('EKPO').select('Purchasing_Document', 'Item', F.col('Order_Quantity').cast('double').alias('_ordered'))
        .join(net_gr, ['Purchasing_Document', 'Item'], 'left').join(seg_open, ['Purchasing_Document', 'Item'], 'left')
        .withColumn('_semantic', F.greatest(F.col('_ordered') - F.coalesce(F.col('_net_gr'), F.lit(0.0)), F.lit(0.0)))
        .withColumn('_diff', F.coalesce(F.col('_seg_open'), F.lit(0.0)) - F.col('_semantic')))
off = cons.filter(F.abs('_diff') > OS_TOLERANCE)
print(f"Items: {cons.count():,} | open quantity differs from the semantic open quantity: {off.count():,} (flagged, not forced)")
off.orderBy(F.abs('_diff').desc()).select('Purchasing_Document', 'Item', '_ordered', '_net_gr', '_seg_open', '_semantic', '_diff').show(20, truncate=False)

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 24, Finished, Available, Finished, False)

Items: 6,039,284 | open quantity differs from the semantic open quantity: 180,340 (flagged, not forced)
+-------------------+----+---------+------------+---------+-----------+------------+
|Purchasing_Document|Item|_ordered |_net_gr     |_seg_open|_semantic  |_diff       |
+-------------------+----+---------+------------+---------+-----------+------------+
|12749930           |1   |9.0E7    |27263248.000|NULL     |6.2736752E7|-6.2736752E7|
|12728978           |3   |1.0E8    |43291846.000|NULL     |5.6708154E7|-5.6708154E7|
|12943014           |1   |7000000.0|NULL        |NULL     |7000000.0  |-7000000.0  |
|13018537           |1   |7000000.0|NULL        |NULL     |7000000.0  |-7000000.0  |
|12943014           |2   |7000000.0|NULL        |NULL     |7000000.0  |-7000000.0  |
|12864700           |1   |5000000.0|NULL        |NULL     |5000000.0  |-5000000.0  |
|13100634           |1   |5000000.0|NULL        |NULL     |5000000.0  |-5000000.0  |
|13018537           |2   |5000000.0|NULL      

In [12]:
# ── 7 · Cases to review with Jeremy ───────────────────────────────────────────────────────────────────
print('Closed lines (rule 3) with no goods receipt (invoice only, or closed without either):')
print(f"  {fact.filter((F.col('Order_Status_Rule') == 3) & F.col('Realized_Finish_Date').isNull()).count():,} rows (questions C6, C14)")

print('\nStock transfers by status (Jeremy reported the lowest agreement here: 76.35%):')
stock_items = silver('EKPO').filter(F.trim(F.col('Item_Category_Ext')) == 'U').select('Purchasing_Document', 'Item').dropDuplicates()   # ▶ 6 Nov 2026: PSTYP U (C4)
(fact.join(stock_items, ['Purchasing_Document', 'Item'], 'left_semi')
     .groupBy('Order_Status_Rule', 'Order_Status_Purchase').count().orderBy('Order_Status_Rule').show(30, truncate=False))

multi = (fact.filter(F.col('Split_Number').isNotNull()).groupBy('Purchasing_Document', 'Item').count().filter('count > 1'))
print(f"Items with several open segments: {multi.count():,} (item-wide goods issue is not copied to each one)")
multi.orderBy(F.desc('count')).show(10, truncate=False)

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 25, Finished, Available, Finished, False)

Closed lines (rule 3) with no goods receipt (invoice only, or closed without either):
  103,610 rows (questions C6, C14)

Stock transfers by status (Jeremy reported the lowest agreement here: 76.35%):
+-----------------+------------------------------+-------+
|Order_Status_Rule|Order_Status_Purchase         |count  |
+-----------------+------------------------------+-------+
|0                |Return item                   |4989   |
|1                |Cancelled partially           |54028  |
|1                |Cancelled completely          |74737  |
|2                |Closed by GR                  |2174989|
|3                |Closed by order creation or IR|4639   |
|4                |Blocked                       |2      |
|9                |Goods issued                  |3402   |
|10               |Outbound delivery created     |1121   |
|14               |Stock committed               |1829   |
|15               |Confirmed                     |108    |
|17               |To be confirm

In [13]:
# ── 8 · Question C4: how is a stock transfer recognised? ──────────────────────────────────────────────────
# ▶ 6 Nov 2026: decided, item category U (Jeremy's SAP screen: U = Stock transfer; L = Subcontracting, so EA's 19 Oct "L" was wrong).
# This compares U (external code EPSTP, Item_Category_Ext) with the earlier test, a supplying plant on the PO header.
header_plant = (silver('EKKO').select('Purchasing_Document', (F.trim(F.coalesce(F.col('Supplying_Plant'), F.lit(''))) != '').alias('Has_Supplying_Plant'))
                .dropDuplicates(['Purchasing_Document']))
items = (silver('EKPO').select('Purchasing_Document', 'Item', F.col('Item_Category_Ext').alias('PSTYP'))   # aliased: the fact has its own Item_Category column
         .join(header_plant, 'Purchasing_Document', 'left'))
print('PO items by item category and supplying plant on the header:')
items.groupBy('PSTYP', 'Has_Supplying_Plant').count().orderBy('PSTYP', 'Has_Supplying_Plant').show(40, truncate=False)
print('Status given, by the same split (rows of the fact):')
(fact.join(items, ['Purchasing_Document', 'Item'], 'left').groupBy('PSTYP', 'Has_Supplying_Plant', 'Order_Status_Purchase').count()
     .orderBy('PSTYP', 'Has_Supplying_Plant', F.desc('count')).show(60, truncate=False))

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 26, Finished, Available, Finished, False)

PO items by item category and supplying plant on the header:
+-----+-------------------+-------+
|PSTYP|Has_Supplying_Plant|count  |
+-----+-------------------+-------+
|     |NULL               |7      |
|     |false              |2941221|
|     |true               |876517 |
|B    |false              |127    |
|D    |false              |104022 |
|D    |true               |1      |
|K    |false              |17654  |
|L    |false              |22461  |
|S    |false              |43027  |
|T    |false              |3      |
|U    |false              |15     |
|U    |true               |2033498|
|W    |false              |731    |
+-----+-------------------+-------+

Status given, by the same split (rows of the fact):
+-----+-------------------+------------------------------+-------+
|PSTYP|Has_Supplying_Plant|Order_Status_Purchase         |count  |
+-----+-------------------+------------------------------+-------+
|     |NULL               |Closed by order creation or IR|7      |
|     

In [14]:
# ── 9 · Questions C6 / C21: delivery completed (EKPO ELIKZ = X) ──────────────────────────────────────────
# ▶ 6 Nov 2026: decided (Jeremy): ELIKZ closes the line (services received), it does not cancel. Expect ELIKZ = X rows in rules 2 / 3
# (or 0 / 1, which come first); ELIKZ = X with any other status points at a problem.
done = silver('EKPO').select('Purchasing_Document', 'Item', (F.col('Delivery_Completed') == 'X').alias('ELIKZ_is_X'))
print('Rows of the fact: ELIKZ = X against closed by quantity (Is_Closed_By_IR is true for every closed line):')
(ev.filter(F.col('Source_Type') == 'EKET').join(done, ['Purchasing_Document', 'Item'], 'left')
     .groupBy('ELIKZ_is_X', 'Is_Closed_By_IR', 'Order_Status_Purchase').count().orderBy('ELIKZ_is_X', 'Is_Closed_By_IR', F.desc('count')).show(60, truncate=False))

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 27, Finished, Available, Finished, False)

Rows of the fact: ELIKZ = X against closed by quantity (Is_Closed_By_IR is true for every closed line):
+----------+---------------+------------------------------+-------+
|ELIKZ_is_X|Is_Closed_By_IR|Order_Status_Purchase         |count  |
+----------+---------------+------------------------------+-------+
|false     |false          |Cancelled completely          |205568 |
|false     |false          |To be confirmed               |52209  |
|false     |false          |Ordered                       |44480  |
|false     |false          |To be ordered                 |14185  |
|false     |false          |Confirmed                     |8573   |
|false     |false          |Goods issued                  |3019   |
|false     |false          |Stock committed               |1829   |
|false     |false          |Return item                   |1621   |
|false     |false          |Cancelled partially           |1518   |
|false     |false          |Blocked                       |1436   |
|false     |

In [15]:
# ── 10 · Questions C17 / C18: goods movements behind GR blocked (103) and Goods issued ─────────────────
# EA 19 Oct: Goods issued = movement 261 (MSEG); GR blocked = a movement 103 (MSEG). The build reads EKBE.
ekbe_all = silver('EKBE')
stock_items = silver('EKPO').filter(F.trim(F.col('Item_Category_Ext')) == 'U').select('Purchasing_Document', 'Item').dropDuplicates()   # ▶ 6 Nov 2026: PSTYP U (C4)
print('EKBE event type (VGABE) and movement type on stock-transfer POs: is there a 261, or 351 / 641, or only VGABE 6?')
(ekbe_all.join(stock_items, ['Purchasing_Document', 'Item'], 'left_semi').groupBy('Transevent_Type', 'Movement_Type').count()
         .orderBy(F.desc('count')).show(30, truncate=False))
print('Blocked-stock postings in EKBE (103 / 107 and their releases 105 / 109):')
(ekbe_all.filter(F.col('Movement_Type').isin('103', '105', '107', '109')).groupBy('Movement_Type', 'DebitCredit_Ind').count()
         .orderBy('Movement_Type').show(truncate=False))
net103 = (ekbe_all.filter(F.col('Movement_Type').isin('103', '105')).groupBy('Purchasing_Document', 'Item')
          .agg(F.sum(F.when(F.col('Movement_Type') == '103', F.col('Signed_Quantity'))).alias('q103'),
               F.sum(F.when(F.col('Movement_Type') == '105', F.col('Signed_Quantity'))).alias('q105')))
n_all = net103.filter(F.col('q103').isNotNull()).count()
n_open = net103.filter(F.col('q103').isNotNull() & (F.coalesce('q103', F.lit(0.0)) - F.coalesce('q105', F.lit(0.0)) > OS_TOLERANCE)).count()
print(f'Items with a 103 posting: {n_all:,}; still blocked after the release 105: {n_open:,}; fully released: {n_all - n_open:,}')
print('The released ones are GR blocked since 22 Oct (EA: a posting is enough); under the earlier rule (net blocked quantity) they were not.')

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 28, Finished, Available, Finished, False)

EKBE event type (VGABE) and movement type on stock-transfer POs: is there a 261, or 351 / 641, or only VGABE 6?
+---------------+-------------+-------+
|Transevent_Type|Movement_Type|count  |
+---------------+-------------+-------+
|1              |101          |2375968|
|8              |             |2314195|
|6              |641          |1920846|
|6              |647          |99479  |
|6              |862          |33325  |
|1              |861          |31441  |
|1              |161          |4764   |
|6              |671          |4735   |
|6              |642          |4310   |
|1              |102          |2166   |
|6              |864          |1878   |
|6              |648          |190    |
|1              |863          |148    |
|1              |162          |22     |
|1              |122          |1      |
+---------------+-------------+-------+

Blocked-stock postings in EKBE (103 / 107 and their releases 105 / 109):
+-------------+---------------+-----+
|Movement_Type|D

In [16]:
# ── 11 · Question C16: does the B1 EKET table have the fields EA 19 Oct names (LIFDT, ETATS)? ─────────────
# EA 19 Oct: Shipping notification = EKET-LIFDT filled and WEMNG = 0. Neither field is in our field lists. Reads the B1 table's columns only.
try:
    b1 = _vl_value(_vl, SOURCE_LH_KEY).rstrip('/')
    eket_cols = spark.read.format('delta').load(f'{b1}/{SOURCE_SCHEMA}/EKET').columns
    for name in ('LIFDT', 'ETATS'):
        print(f"  EKET.{name:<6} {'present in B1' if name in eket_cols else 'NOT in B1: the 19 Oct condition cannot be built'}")
except Exception as e:
    print(f'  B1 EKET could not be read ({str(e)[:100]}); list its columns by hand: spark.read.format("delta").load(<B1>/Tables/EKET).columns')

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 29, Finished, Available, Finished, False)

  EKET.LIFDT  NOT in B1: the 19 Oct condition cannot be built
  EKET.ETATS  NOT in B1: the 19 Oct condition cannot be built


In [17]:
# ── 12 · Movement 107 (advance shipping notice, Jeremy 6 Nov): does it raise the received quantity (EKET WEMNG)? ──
# The status treats an open 107 (107 minus 109 above 0) as Shipping notification. If WEMNG already counts the 107 quantity, the
# line can look received (closed) before the goods arrive; then the net 107 quantity must be taken out of the received quantity.
_e = silver('EKBE')
asn = (_e.filter(F.col('Movement_Type').isin('107', '109')).groupBy('Purchasing_Document', 'Item')
         .agg((F.sum(F.when(F.col('Movement_Type') == '107', F.col('Signed_Quantity')))
               - F.coalesce(F.sum(F.when(F.col('Movement_Type') == '109', F.col('Signed_Quantity'))), F.lit(0.0))).alias('Net_107'))
         .filter(F.col('Net_107') > OS_TOLERANCE))
gr_other = (_e.filter(((F.col('Transevent_Type') == '1') | (F.col('PO_History_Category') == 'E')) & ~F.col('Movement_Type').isin('107', '109'))
              .groupBy('Purchasing_Document', 'Item').agg(F.sum('Signed_Quantity').alias('Other_GR')))
wemng = silver('EKET').groupBy('Purchasing_Document', 'Item').agg(F.sum('Quantity_Delivered').alias('WEMNG'))
cmp = (asn.join(gr_other, ['Purchasing_Document', 'Item'], 'left').join(wemng, ['Purchasing_Document', 'Item'], 'left')
          .withColumn('WEMNG_includes_107', F.abs(F.coalesce('WEMNG', F.lit(0.0)) - F.coalesce('Other_GR', F.lit(0.0)) - F.col('Net_107')) <= OS_TOLERANCE))
print(f"Items with an open 107: {cmp.count():,}")
cmp.groupBy('WEMNG_includes_107').count().show()
print('Mostly true: WEMNG counts the 107, so the build must take it out of the received quantity (tell the developer). Mostly false: nothing to do.')
cmp.show(10, truncate=False)

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 30, Finished, Available, Finished, False)

Items with an open 107: 0
+------------------+-----+
|WEMNG_includes_107|count|
+------------------+-----+
+------------------+-----+

Mostly true: WEMNG counts the 107, so the build must take it out of the received quantity (tell the developer). Mostly false: nothing to do.
+-------------------+----+-------+--------+-----+------------------+
|Purchasing_Document|Item|Net_107|Other_GR|WEMNG|WEMNG_includes_107|
+-------------------+----+-------+--------+-----+------------------+
+-------------------+----+-------+--------+-----+------------------+



In [18]:
# ── 13 · Question C20: return items and negative quantities ──────────────────────────────────────────────
# Jeremy: return POs are flagged (EKPO RETPO = X: rule 0 Return item, excluded in the reports); any other negative quantity is an error.
print(f"Return-item rows (rule 0): {fact.filter(F.col('Order_Status_Rule') == 0).count():,}")
neg = (silver('EKET').filter(F.col('Scheduled_Quantity') < 0).select('Purchasing_Document', 'Item').dropDuplicates()
       .join(silver('EKPO').select('Purchasing_Document', 'Item', 'Returns_Item'), ['Purchasing_Document', 'Item'], 'left'))
n_bad = neg.filter(F.trim(F.coalesce(F.col('Returns_Item'), F.lit(''))) != 'X').count()
print(f"Items with a negative scheduled quantity: {neg.count():,}; not return items (data errors per Jeremy): {n_bad:,}")
dq_record('CHECK', 'Fact_PO_LineItem_Schedule', 'negative_not_return', 'WARN' if n_bad else 'PASS',
          f'{n_bad:,} items have a negative scheduled quantity but are not return items', observed=n_bad)

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 31, Finished, Available, Finished, False)

Return-item rows (rule 0): 24,851
Items with a negative scheduled quantity: 0; not return items (data errors per Jeremy): 0


{'pipeline_run_id': 'edf70372-e7c0-43c7-a6fd-e89a04dbaf87',
 'pipeline_name': 'PROC_Order_Status_Check',
 'layer': 'CHECK',
 'table_name': 'Fact_PO_LineItem_Schedule',
 'column_name': None,
 'check_name': 'negative_not_return',
 'abc_category': 'CONTROL',
 'severity': 'WARN',
 'status': 'PASS',
 'observed': 0.0,
 'expected': None,
 'threshold': None,
 'message': '0 items have a negative scheduled quantity but are not return items',
 'logged_at': datetime.datetime(2026, 11, 19, 21, 32, 33, 42116)}

In [19]:
# ── 14 · Question C12: confirmation allocation (6 Nov 2026) ───────────────────────────────────────────────
# Open EKES confirmations are spread over the item's open schedule lines in date order. Per item: confirmed pieces plus line
# remainders must equal what is still open on the open lines (scheduled minus received). Over-confirmed pieces have no line.
# ▶ 15 Nov 2026: open lines from every row that names a line (a fully confirmed line has only piece rows now)
_open_lines = (silver('EKET').join(ev.filter(F.col('Schedule_Line').isNotNull() & ~F.col('Order_Status_Rule').isin(0, 1, 2, 3))
                                     .select('Purchasing_Document', 'Item', 'Schedule_Line').dropDuplicates(),
                                   ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left_semi')
               .groupBy('Purchasing_Document', 'Item')
               .agg(F.sum(F.greatest(F.col('Scheduled_Quantity') - F.coalesce(F.col('Quantity_Delivered'), F.lit(0.0)), F.lit(0.0))).alias('_open')))
_segs = (ev.filter(~F.coalesce(F.col('Is_Over_Confirmed'), F.lit(False)) & ~F.col('Order_Status_Rule').isin(0, 1, 2, 3))
             .groupBy('Purchasing_Document', 'Item').agg(F.sum(F.col('Segment_Quantity').cast('double')).alias('_segs')))
rec = _open_lines.join(_segs, ['Purchasing_Document', 'Item'], 'full').fillna(0.0).withColumn('_diff', F.col('_segs') - F.col('_open'))
print(f"Items: {rec.count():,} | pieces + remainders differ from the open quantity: {rec.filter(F.abs('_diff') > OS_TOLERANCE).count():,} (expected 0)")
print(f"Confirmed pieces: {ev.filter(F.col('Source_Type') == 'EKES').count():,}; over-confirmed: "   # ▶ 17 Nov 2026: from the evidence table
      f"{ev.filter(F.coalesce(F.col('Is_Over_Confirmed'), F.lit(False))).count():,}; lines fully confirmed: "
      f"{ev.filter(F.col('Is_Line_Fully_Confirmed')).select('Purchasing_Document', 'Item', 'Schedule_Line').distinct().count():,}")
print('Lines split between confirmed and unconfirmed (Jeremy\'s half-and-half case):')
(ev.filter(F.col('Source_Type') == 'EKES').select('Purchasing_Document', 'Item', 'Schedule_Line').dropDuplicates()
     .join(ev.filter((F.col('Source_Type') == 'EKET') & (F.col('Segment_Quantity') > 0)).select('Purchasing_Document', 'Item', 'Schedule_Line'),
           ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left_semi').count())   # ▶ 17 Nov 2026: segments are in the evidence table

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 32, Finished, Available, Finished, False)

Items: 151,750 | pieces + remainders differ from the open quantity: 0 (expected 0)
Confirmed pieces: 47,368; over-confirmed: 18,414; lines fully confirmed: 27,635
Lines split between confirmed and unconfirmed (Jeremy's half-and-half case):


197

In [20]:
# ── 15 · Question C22: lines closed by the under-delivery tolerance (EKPO UNTTO) ───────────────────────────
# Jeremy: a line is complete when the rest is within the receiving tolerance. These lines are closed only because of it.
_t = (silver('EKET').join(silver('EKPO').select('Purchasing_Document', 'Item', 'Underdelivery_Tolerance'), ['Purchasing_Document', 'Item'], 'left')
        .withColumn('_open', F.col('Scheduled_Quantity') - F.coalesce(F.col('Quantity_Delivered'), F.lit(0.0)))
        .filter((F.col('_open') > OS_TOLERANCE)
                & (F.col('_open') <= F.col('Scheduled_Quantity') * F.coalesce(F.col('Underdelivery_Tolerance').cast('double'), F.lit(0.0)) / 100)))
print(f"Schedule lines closed by the tolerance only: {_t.count():,}")
_t.groupBy('Underdelivery_Tolerance').count().orderBy(F.desc('count')).show(10)

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 33, Finished, Available, Finished, False)

Schedule lines closed by the tolerance only: 15,175
+-----------------------+-----+
|Underdelivery_Tolerance|count|
+-----------------------+-----+
|                   20.0| 9357|
|                   10.0| 4403|
|                    5.0| 1101|
|                    1.0|  304|
|                   30.0|    3|
|                   80.0|    2|
|                   50.0|    2|
|                   45.0|    1|
|                   15.0|    1|
|                   26.0|    1|
+-----------------------+-----+



In [21]:
# ── End-of-run report: result, issues in plain language, tables, checks ────────
end_of_run_report()

StatementMeta(, 77d35642-3dc7-4393-bbf1-64c4c37d7711, 34, Finished, Available, Finished, False)

════════════════════════════════════════════════════════════════════════════════════════════════════
 PROC_Order_Status_Check  |  run edf70372-e7c0-43c7-a6fd-e89a04dbaf87
 19 Nov 2026 21:28–21:33 UTC (4m 38s)  |  0 of 0 tables OK  |  1 checks
 RESULT: ✓ All 0 tables loaded, all checks passed
════════════════════════════════════════════════════════════════════════════════════════════════════
 CHECKS PER TABLE
 Table                      Passed  Issues
 Fact_PO_LineItem_Schedule       1  —

 Details: SELECT * FROM dq_results WHERE pipeline_run_id = 'edf70372-e7c0-43c7-a6fd-e89a04dbaf87' ORDER BY status
════════════════════════════════════════════════════════════════════════════════════════════════════


SynapseWidget(Synapse.DataFrame, 1337d84b-d9ab-4129-b451-cf38ff0206b2)

Session stopped to release capacity (END_SESSION = False keeps it)
  Session not stopped (__$$_STOP_SESSION_REQUEST_$$__)


'SUCCEEDED'